# 00 - rsl_rl 手撕路线图

这套 notebook 的目标不是让你“看懂一些代码”, 而是把你训练到能独立维护和重写 rsl_rl 的核心结构。

学习顺序:

1. `01_torch_minimum.ipynb`: 先把 PyTorch 的 tensor、Module、loss、backward、optimizer 讲透。
2. `02_ppo_dataflow_and_gae.ipynb`: 手算 PPO 数据流和 GAE。
3. `03_actor_critic_distribution.ipynb`: 手写 actor、critic、高斯分布、log_prob、entropy。
4. `04_rollout_storage_toy.ipynb`: 手写一个玩具 RolloutStorage。
5. `05_multi_ppo_concepts.ipynb`: 手写 multi-critic 的 reward/value/advantage 形状。
6. `06_amp_reward_flow.ipynb`: 手写 AMP 的 task/style/final reward 逻辑。

每节课的验收标准都很具体: 你要能预测 shape, 解释梯度, 改一行代码并知道会发生什么。

## rsl_rl 主线地图

运行顺序不是文件顺序。真正的执行路径是:

```text
OnPolicyRunner.learn
-> PPO.act
-> env.step
-> PPO.process_env_step
-> RolloutStorage.add_transition
-> PPO.compute_returns
-> PPO.update
```

你以后读任何 RL 仓库, 都先找这条线。

## 文件职责

- `rsl_rl/runners/on_policy_runner.py`: 训练流程调度。
- `rsl_rl/algorithms/ppo.py`: PPO 数学和优化。
- `rsl_rl/storage/rollout_storage.py`: rollout 数据布局和 mini-batch 生成。
- `rsl_rl/models/mlp_model.py`: actor/critic 网络 forward。
- `rsl_rl/modules/distribution.py`: action distribution、sample、log_prob、entropy、KL。
- `rsl_rl/algorithms/multi_ppo.py`: multi-critic PPO。
- `rsl_rl/algorithms/amp_ppo.py`: AMP reward/discriminator 逻辑。

核心边界:

```text
runner 管流程
algorithm 管训练数学
storage 管数据形状
model 管 forward
builder 管对象构造
```

## 第一条训练规矩

每看到一个 tensor, 你必须问:

```text
shape 是什么?
谁创建的?
谁消费它?
它带梯度吗?
它属于 old policy 还是 current policy?
```